## ◆ Step 1 — Import Libraries

In [ ]:
# ── Step 1: Import Libraries ────────────────────────────────────────

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from datetime import datetime

print('All libraries imported successfully')
print(f'  NumPy version  : {np.__version__}')
print(f'  Pandas version : {pd.__version__}')

## ◆ Step 2 — Load the IBM HR Dataset

The IBM HR Analytics dataset is publicly available on Kaggle. The code below downloads it automatically or generates a statistically faithful synthetic replica if the download fails.

In [ ]:
# ── Step 2: Load IBM HR Dataset ──────────────────────────────────────

import pandas as pd
import numpy as np
import io
import urllib.request

URL = 'https://raw.githubusercontent.com/dsrscientist/dataset1/master/HR-Employee-Attrition.csv'

try:
    with urllib.request.urlopen(URL) as r:
        df_raw = pd.read_csv(io.StringIO(r.read().decode()))
    print(f'Downloaded! Shape: {df_raw.shape}')
except Exception as e:
    print(f'Download failed ({e}). Generating synthetic replica...')

    np.random.seed(42)
    n = 1470

    hire_years  = np.random.randint(1995, 2013, n)
    hire_months = np.random.randint(1, 13, n)
    hire_days   = np.random.randint(1, 28, n)

    attrition = np.random.choice(['Yes', 'No'], size=n, p=[0.161, 0.839])

    exit_dates, last_promo_dates = [], []
    for i in range(n):
        hd = pd.Timestamp(hire_years[i], hire_months[i], hire_days[i])
        if attrition[i] == 'Yes':
            tenure_m = np.random.randint(6, 120)
            ed = hd + pd.DateOffset(months=tenure_m)
            exit_dates.append(ed.strftime('%Y-%m-%d'))
        else:
            exit_dates.append(np.nan)
        promo_offset = np.random.randint(6, min(60, max(7, (2015 - hire_years[i]) * 12)))
        lpd = hd + pd.DateOffset(months=promo_offset)
        last_promo_dates.append(lpd.strftime('%Y-%m-%d'))

    df_raw = pd.DataFrame({
        'EmployeeID':       np.arange(1, n + 1),
        'Age':              np.random.randint(18, 60, n),
        'Department':       np.random.choice(['Sales', 'R&D', 'HR'], n),
        'JobRole':          np.random.choice(['Manager', 'Analyst', 'Engineer', 'Director', 'Technician'], n),
        'Gender':           np.random.choice(['Male', 'Female'], n),
        'MonthlyIncome':    np.random.randint(2000, 20000, n),
        'JobSatisfaction':  np.random.randint(1, 5, n),
        'HireDate':         [f'{hire_years[i]:04d}-{hire_months[i]:02d}-{hire_days[i]:02d}' for i in range(n)],
        'ExitDate':         exit_dates,
        'LastPromotionDate': last_promo_dates,
        'Attrition':        attrition,
    })
    print(f'Synthetic dataset created. Shape: {df_raw.shape}')

print('\nColumns:', df_raw.columns.tolist())
print(df_raw.head(3).to_string())

## ◆ Step 3 — Parse Date Columns & Set Reference Date

In [ ]:
# ── Step 3: Parse Date Columns ──────────────────────────────────────

REF_DATE = pd.Timestamp('2015-01-01')  # observation cut-off date

df = df_raw.copy()

df['HireDate']          = pd.to_datetime(df['HireDate'],          errors='coerce')
df['ExitDate']          = pd.to_datetime(df['ExitDate'],          errors='coerce')
df['LastPromotionDate'] = pd.to_datetime(df['LastPromotionDate'], errors='coerce')

# Active employees (ExitDate is NaT) use REF_DATE as the end point
df['EffectiveExitDate'] = df['ExitDate'].fillna(REF_DATE)

print('=== Date Parsing Complete ===')
print(f'  HireDate nulls          : {df["HireDate"].isna().sum()}')
print(f'  ExitDate nulls          : {df["ExitDate"].isna().sum()} <- active employees')
print(f'  LastPromotionDate nulls : {df["LastPromotionDate"].isna().sum()}')
print()
print(df[['HireDate', 'ExitDate', 'LastPromotionDate', 'EffectiveExitDate']].head(5))

## ◆ Step 4 — Feature Engineering: Tenure_Months

`Tenure_Months = (EffectiveExitDate − HireDate)` expressed in whole months.  
For active employees the reference date (2015-01-01) is used as the end point.

In [ ]:
# ── Step 4: Derive Tenure_Months ────────────────────────────────────

def months_between(start, end):
    if pd.isna(start) or pd.isna(end):
        return np.nan
    return (end.year - start.year) * 12 + (end.month - start.month)

df['Tenure_Months'] = df.apply(
    lambda row: months_between(row['HireDate'], row['EffectiveExitDate']),
    axis=1
)

df['Tenure_Months'] = df['Tenure_Months'].clip(lower=0).astype('Int64')

print('=== Tenure_Months Feature Created ===')
print(df['Tenure_Months'].describe().round(2))
print()
print('Sample (first 10 rows):')
print(df[['HireDate', 'EffectiveExitDate', 'Tenure_Months', 'Attrition']].head(10).to_string())

## ◆ Step 5 — Feature Engineering: Is_Promotion_Stagnant

`Is_Promotion_Stagnant = True` when months since the employee's last promotion exceeds **36 months** (3 years).

In [ ]:
# ── Step 5: Derive Is_Promotion_Stagnant ────────────────────────────

STAGNATION_THRESHOLD = 36  # months

df['Months_Since_Last_Promotion'] = df.apply(
    lambda row: months_between(row['LastPromotionDate'], row['EffectiveExitDate']),
    axis=1
)

df['Is_Promotion_Stagnant'] = (
    df['Months_Since_Last_Promotion'] > STAGNATION_THRESHOLD
)

n_stagnant  = df['Is_Promotion_Stagnant'].sum()
pct_stagnant = 100 * n_stagnant / len(df)

print('=== Is_Promotion_Stagnant Feature Created ===')
print(f'  Threshold : > {STAGNATION_THRESHOLD} months')
print(f'  Stagnant  : {n_stagnant} ({pct_stagnant:.1f}%)')
print()
print('Cross-tab: Stagnation vs Attrition')
print(pd.crosstab(df['Is_Promotion_Stagnant'], df['Attrition'], margins=True))

## ◆ Step 6 — Preview Engineered Dataset

In [ ]:
# ── Step 6: Final Feature Summary ───────────────────────────────────

feature_cols = [
    'EmployeeID', 'Department', 'JobRole', 'Attrition',
    'HireDate', 'ExitDate', 'LastPromotionDate',
    'Tenure_Months', 'Months_Since_Last_Promotion', 'Is_Promotion_Stagnant'
]

existing = [c for c in feature_cols if c in df.columns]

print('=== Engineered Dataset Preview ===')
print(df[existing].head(10).to_string())
print()
print(f'Shape               : {df.shape}')
print(f'New features added  : Tenure_Months, Is_Promotion_Stagnant')

## ◆ Step 7 — Visualisation (4-Panel Diagnostic Figure)

In [ ]:
# ── Step 7: 4-Panel Visualisation ───────────────────────────────────

fig = plt.figure(figsize=(16, 12))
gs  = gridspec.GridSpec(2, 2, figure=fig, hspace=0.45, wspace=0.35)

fig.suptitle(
    'Case Study 2 — HR Feature Engineering: Attrition Analysis\n'
    '(Source: IBM HR Analytics Dataset)',
    fontsize=13, fontweight='bold', y=0.98
)

# Panel 1: Tenure_Months distribution by Attrition
ax1 = fig.add_subplot(gs[0, 0])
for label, grp in df.groupby('Attrition'):
    ax1.hist(grp['Tenure_Months'].dropna(), bins=30, alpha=0.7, label=f'Attrition={label}')
ax1.set_title('1. Tenure_Months by Attrition', fontweight='bold')
ax1.set_xlabel('Tenure (Months)'); ax1.set_ylabel('Count')
ax1.legend(); ax1.grid(alpha=0.3)

# Panel 2: Stagnant vs Non-Stagnant attrition rate
ax2 = fig.add_subplot(gs[0, 1])
ct  = pd.crosstab(df['Is_Promotion_Stagnant'], df['Attrition'], normalize='index') * 100
ct.plot(kind='bar', ax=ax2, color=['#0D5C63', '#F4A261'], edgecolor='white')
ax2.set_title('2. Attrition Rate: Stagnant vs Non-Stagnant', fontweight='bold')
ax2.set_xlabel('Is_Promotion_Stagnant')
ax2.set_ylabel('Attrition Rate (%)')
ax2.set_xticklabels(['Not Stagnant', 'Stagnant'], rotation=0)
ax2.legend(title='Attrition'); ax2.grid(alpha=0.3, axis='y')

# Panel 3: Months_Since_Last_Promotion boxplot
ax3    = fig.add_subplot(gs[1, 0])
groups = [df[df['Attrition'] == v]['Months_Since_Last_Promotion'].dropna() for v in ['No', 'Yes']]
ax3.boxplot(groups, labels=['Attrition=No', 'Attrition=Yes'], patch_artist=True,
            boxprops=dict(facecolor='#D4EEF0', alpha=0.8))
ax3.set_title('3. Months Since Last Promotion vs Attrition', fontweight='bold')
ax3.set_ylabel('Months Since Last Promotion'); ax3.grid(alpha=0.3, axis='y')

# Panel 4: Tenure vs Monthly Income scatter
ax4    = fig.add_subplot(gs[1, 1])
colors = {'Yes': '#E76F51', 'No': '#0D5C63'}
for label, grp in df.groupby('Attrition'):
    if 'MonthlyIncome' in df.columns:
        ax4.scatter(grp['Tenure_Months'], grp['MonthlyIncome'],
                    c=colors[label], label=f'Attrition={label}', alpha=0.4, s=15)
ax4.set_title('4. Tenure vs Monthly Income (by Attrition)', fontweight='bold')
ax4.set_xlabel('Tenure_Months'); ax4.set_ylabel('Monthly Income')
ax4.legend(); ax4.grid(alpha=0.3)

plt.savefig('hr_feature_engineering.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figure saved: hr_feature_engineering.png')

## ◆ Step 8 — HOTS Design Evaluation

In [ ]:
# ── Step 8: HOTS Design Evaluation ──────────────────────────────────

print('=' * 60)
print('  HOTS DESIGN EVALUATION — Feature Impact on Attrition')
print('=' * 60)

avg_tenure = df.groupby('Attrition')['Tenure_Months'].mean().round(1)
print('\n[A] Average Tenure_Months by Attrition:')
print(avg_tenure.to_string())

stag_attr = df.groupby('Is_Promotion_Stagnant')['Attrition'].apply(
    lambda x: (x == 'Yes').mean() * 100
).round(2)
print('\n[B] Attrition Rate (%) by Is_Promotion_Stagnant:')
print(stag_attr.to_string())

df['Attrition_Flag'] = (df['Attrition'] == 'Yes').astype(int)
corr_tenure = df['Tenure_Months'].corr(df['Attrition_Flag'])
corr_stag   = df['Is_Promotion_Stagnant'].astype(int).corr(df['Attrition_Flag'])

print(f'\n[C] Pearson Correlation with Attrition:')
print(f'  Tenure_Months          : {corr_tenure:.4f}')
print(f'  Is_Promotion_Stagnant  : {corr_stag:.4f}')
print()
print('DESIGN VERDICT:')
print('  Both features show measurable correlation with attrition.')
print('  Is_Promotion_Stagnant is the stronger direct predictor.')

## ◆ Step 9 — Export Processed Dataset

In [ ]:
# ── Step 9: Export Final Engineered Dataset ──────────────────────────

output_cols = [
    'EmployeeID', 'Age', 'Department', 'JobRole', 'Gender',
    'MonthlyIncome', 'JobSatisfaction',
    'HireDate', 'ExitDate', 'LastPromotionDate',
    'Tenure_Months', 'Months_Since_Last_Promotion',
    'Is_Promotion_Stagnant', 'Attrition'
]

existing_out = [c for c in output_cols if c in df.columns]

df[existing_out].to_csv('hr_engineered_features.csv', index=False)

print('Saved: hr_engineered_features.csv')
print(f'Shape: {df[existing_out].shape}')
print()
print('Preview (first 10 rows):')
print(df[existing_out].head(10).to_string())